# PaperLab AI Detector V1 — Model Training Pipeline

A clean, end-to-end training notebook for **PaperLab AI Detector V1** based on `microsoft/deberta-v3-small`.

### Objectives:
- Train a lightweight sequence classifier (`0: Human`, `1: AI`).
- Combine **DAIGT V3** (persuasive/academic student essays) + **HC3** (computer science, medicine, finance, open QA).
- Implement **Generator-Disjoint Splitting**: evaluate on unseen AI models (`Falcon-180B`, `LLaMA-70B`) to measure true generalization.
- Optimize training for **RTX 4050 (6GB VRAM)** using `fp16` mixed precision and gradient accumulation.
- Export production-ready model to `models/paperlab-detector-v1/`.

> **Note:** Follow the cells in sequence. You can adjust sample sizes, epochs, or batch sizes directly in the configuration cell.

## 1. Environment & Hardware Verification
Verify PyTorch version, CUDA GPU access, and available VRAM.

In [ ]:
import os
import torch
import transformers
import pandas as pd
import numpy as np

print(f"PyTorch Version: {torch.__version__}")
print(f"Transformers Version: {transformers.__version__}")

cuda_available = torch.cuda.is_available()
print(f"CUDA Available: {cuda_available}")

if cuda_available:
    device_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Device: {device_name} ({total_mem:.2f} GB VRAM)")
else:
    print("Running on CPU.")


## 2. Configuration & Hyperparameters
Fine-tuned for RTX 4050 6GB VRAM (`fp16`, batch size 8 with gradient accumulation 2 = effective batch size 16).

In [ ]:
CONFIG = {
    "model_name": "microsoft/deberta-v3-small",
    "max_length": 512,
    "learning_rate": 2e-5,
    "train_batch_size": 8,
    "eval_batch_size": 16,
    "gradient_accumulation_steps": 2,
    "num_train_epochs": 3,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "fp16": True if torch.cuda.is_available() else False,
    "random_seed": 42,
    "max_train_samples": 30000,  # Cap training samples for fast iteration (set None for full dataset)
    "output_dir": "models/checkpoints",
    "final_model_dir": "models/paperlab-detector-v1",
}

print("Training configuration loaded:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")


## 3. Data Ingestion & Standardization
Loads DAIGT V3 and HC3 subsets from `dataset/raw/`, cleans whitespace, filters length, and maps them to a unified schema.

In [ ]:
import json
from pathlib import Path

DATASET_ROOT = Path("dataset/raw")
records = []

# 1. Load DAIGT
daigt_path = DATASET_ROOT / "daigt" / "daigt_v3_drcat.csv"
if daigt_path.exists():
    print(f"Loading DAIGT from {daigt_path}...")
    df_daigt = pd.read_csv(daigt_path)
    for _, row in df_daigt.iterrows():
        text = str(row.get("text", "")).strip()
        if len(text.split()) < 30:
            continue
        records.append({
            "text": text,
            "label": int(row["label"]),
            "source": "daigt",
            "generator": str(row.get("source", "unknown")),
            "domain": "academic_essay"
        })
    print(f"  Loaded {len(records)} valid samples from DAIGT.")

# 2. Load HC3 Subsets
hc3_dir = DATASET_ROOT / "hc3"
hc3_files = {
    "wiki_csai.jsonl": "computer_science",
    "medicine.jsonl": "medicine",
    "finance.jsonl": "finance",
    "open_qa.jsonl": "open_qa"
}

hc3_count = 0
for filename, domain in hc3_files.items():
    filepath = hc3_dir / filename
    if not filepath.exists():
        continue
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            item = json.loads(line)
            for ans in item.get("human_answers", []):
                ans_clean = ans.strip()
                if len(ans_clean.split()) >= 30:
                    records.append({
                        "text": ans_clean,
                        "label": 0,
                        "source": "hc3",
                        "generator": "human",
                        "domain": domain
                    })
                    hc3_count += 1
            for ans in item.get("chatgpt_answers", []):
                ans_clean = ans.strip()
                if len(ans_clean.split()) >= 30:
                    records.append({
                        "text": ans_clean,
                        "label": 1,
                        "source": "hc3",
                        "generator": "chatgpt",
                        "domain": domain
                    })
                    hc3_count += 1

print(f"  Loaded {hc3_count} valid samples from HC3.")

df_all = pd.DataFrame(records)
df_all = df_all.drop_duplicates(subset=["text"]).reset_index(drop=True)
print(f"\nTotal unique dataset size: {len(df_all)} samples")
print("Overall Label Distribution:")
print(df_all["label"].value_counts().rename({0: "0 (Human)", 1: "1 (AI)"}))


## 4. Generator-Disjoint Splitting
As specified in Section 8 & 25 of `readme.md`, we strictly isolate unseen generator models (`falcon_180b`, `llama_70b`) for the **Test set** to test genuine out-of-distribution generalization rather than memorization.

In [ ]:
from sklearn.model_selection import train_test_split

# Designate unseen test generators for out-of-distribution evaluation
UNSEEN_TEST_GENERATORS = [
    "llama_falcon_v3_falcon_180b",
    "llama_falcon_v3_llama_70b"
]

# Separate unseen generator samples for testing
test_mask = df_all["generator"].isin(UNSEEN_TEST_GENERATORS)
df_test_unseen = df_all[test_mask].copy()
df_pool = df_all[~test_mask].copy()

# Sample balanced human text for the test set
df_human_pool = df_pool[df_pool["label"] == 0]
df_ai_pool = df_pool[df_pool["label"] == 1]

human_test_sample = df_human_pool.sample(n=min(len(df_human_pool), len(df_test_unseen)), random_state=CONFIG["random_seed"])
df_test = pd.concat([df_test_unseen, human_test_sample]).sample(frac=1.0, random_state=CONFIG["random_seed"]).reset_index(drop=True)

# Remaining samples for train and validation
remaining_pool = df_pool.drop(index=human_test_sample.index).reset_index(drop=True)

df_train, df_val = train_test_split(
    remaining_pool,
    test_size=0.10,
    stratify=remaining_pool["label"],
    random_state=CONFIG["random_seed"]
)

# Optionally cap training set for fast local training
if CONFIG["max_train_samples"] and len(df_train) > CONFIG["max_train_samples"]:
    df_train = df_train.sample(n=CONFIG["max_train_samples"], random_state=CONFIG["random_seed"]).reset_index(drop=True)

df_val = df_val.reset_index(drop=True)

print(f"Train set:      {len(df_train)} samples (Human: {(df_train['label']==0).sum()}, AI: {(df_train['label']==1).sum()})")
print(f"Validation set: {len(df_val)} samples (Human: {(df_val['label']==0).sum()}, AI: {(df_val['label']==1).sum()})")
print(f"Test set (Unseen Models): {len(df_test)} samples (Human: {(df_test['label']==0).sum()}, AI: {(df_test['label']==1).sum()})")


## 5. Tokenization & Dataset Preparation
Load DeBERTa-v3-small tokenizer and tokenize inputs up to 512 tokens.

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

print(f"Loading tokenizer: {CONFIG['model_name']}...")
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        max_length=CONFIG["max_length"],
        truncation=True,
        padding=False  # Dynamic padding will be handled by DataCollator
    )

train_ds = Dataset.from_pandas(df_train[["text", "label"]])
val_ds = Dataset.from_pandas(df_val[["text", "label"]])
test_ds = Dataset.from_pandas(df_test[["text", "label"]])

print("Tokenizing datasets...")
tokenized_train = train_ds.map(tokenize_batch, batched=True, remove_columns=["text"])
tokenized_val = val_ds.map(tokenize_batch, batched=True, remove_columns=["text"])
tokenized_test = test_ds.map(tokenize_batch, batched=True, remove_columns=["text"])

print("Tokenization complete.")


## 6. Evaluation Metrics: Accuracy, AUROC & False Positive Rate
Section 19 of `readme.md` emphasizes that **False Positive Rate (FPR)** on human writing must be tracked alongside Accuracy and AUROC.

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix
from scipy.special import softmax

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = softmax(logits, axis=1)[:, 1]  # P(AI)
    preds = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
    
    try:
        auroc = roc_auc_score(labels, probs)
    except Exception:
        auroc = 0.0

    cm = confusion_matrix(labels, preds, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    return {
        "accuracy": round(acc, 4),
        "f1": round(f1, 4),
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "auroc": round(auroc, 4),
        "false_positive_rate": round(fpr, 4),
        "false_negative_rate": round(fnr, 4)
    }


## 7. Model Initialization & Trainer Setup
Instantiate DeBERTa-v3-small classification head and prepare the Hugging Face `Trainer`.

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding

print(f"Initializing {CONFIG['model_name']} with 2 classification labels...")
model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG["model_name"],
    num_labels=2,
    id2label={0: "HUMAN", 1: "AI"},
    label2id={"HUMAN": 0, "AI": 1}
)

training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=CONFIG["learning_rate"],
    per_device_train_batch_size=CONFIG["train_batch_size"],
    per_device_eval_batch_size=CONFIG["eval_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    num_train_epochs=CONFIG["num_train_epochs"],
    weight_decay=CONFIG["weight_decay"],
    warmup_ratio=CONFIG["warmup_ratio"],
    fp16=CONFIG["fp16"],
    metric_for_best_model="f1",
    load_best_model_at_end=True,
    logging_steps=50,
    save_total_limit=2,
    report_to="none",
    seed=CONFIG["random_seed"]
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer configured and ready to run!")


## 8. Train the Model
Execute the cell below when you are ready to start training.

In [ ]:
# Run training
print("Starting training...")
train_result = trainer.train()
print("Training completed successfully!")
print(train_result.metrics)


## 9. Evaluation on Unseen Generators (True Generalization Benchmark)
Evaluate the trained model on unseen AI models (`Falcon-180B`, `LLaMA-70B`) to check generalization.

In [ ]:
print("=== EVALUATION: IN-DISTRIBUTION VALIDATION SET ===")
val_metrics = trainer.evaluate(eval_dataset=tokenized_val)
for k, v in val_metrics.items():
    print(f"  {k}: {v}")

print("\n=== EVALUATION: OUT-OF-DISTRIBUTION UNSEEN GENERATORS TEST SET ===")
test_metrics = trainer.evaluate(eval_dataset=tokenized_test)
for k, v in test_metrics.items():
    print(f"  {k}: {v}")


## 10. Save Final Model & Tokenizer
Export the best checkpoint to `models/paperlab-detector-v1/` for production inference.

In [ ]:
final_dir = Path(CONFIG["final_model_dir"])
final_dir.mkdir(parents=True, exist_ok=True)

print(f"Saving final model to {final_dir}...")
trainer.save_model(str(final_dir))
tokenizer.save_pretrained(str(final_dir))

# Save training metadata and metrics for reproducibility
meta = {
    "config": CONFIG,
    "val_metrics": {k: float(v) for k, v in val_metrics.items() if isinstance(v, (int, float))},
    "test_metrics": {k: float(v) for k, v in test_metrics.items() if isinstance(v, (int, float))}
}
with open(final_dir / "training_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print(f"Export complete! Files saved in {final_dir}:")
for f in final_dir.iterdir():
    print(f" - {f.name} ({f.stat().st_size / (1024*1024):.2f} MB)")


## 11. Document-Level Inference & Chunk Aggregation Test
Test the trained detector with a sample text to produce the structured JSON output specified in Section 21 of `readme.md`.

In [ ]:
import torch.nn.functional as F

def predict_document(text, chunk_size_words=150):
    """
    Chunks document text and returns structured probability output.
    """
    words = text.split()
    chunks = [" ".join(words[i:i + chunk_size_words]) for i in range(0, len(words), chunk_size_words)]
    if not chunks:
        return {"error": "Empty text"}

    model.eval()
    device = next(model.parameters()).device

    chunk_results = []
    scores = []

    for idx, chunk in enumerate(chunks):
        inputs = tokenizer(chunk, return_tensors="pt", truncation=True, max_length=512).to(device)
        with torch.no_grad():
            logits = model(**inputs).logits
            probs = F.softmax(logits, dim=-1)[0].cpu().numpy()
            p_ai = float(probs[1])
            scores.append(p_ai)
            chunk_results.append({
                "index": idx,
                "words": len(chunk.split()),
                "ai_probability": round(p_ai, 4)
            })

    doc_score = float(np.mean(scores))
    if doc_score < 0.35:
        classification = "likely_human"
    elif doc_score > 0.65:
        classification = "likely_ai"
    else:
        classification = "uncertain"

    return {
        "document_score": round(doc_score, 4),
        "classification": classification,
        "num_chunks": len(chunks),
        "chunks": chunk_results
    }

# Sample Test
sample_academic_text = (
    "This study investigates the comparative efficiency of transformer-based architectures "
    "for domain-specific classification. We evaluate DeBERTa-v3 on curated academic corpora, "
    "analyzing out-of-distribution generalization across unseen generative models. Our empirical "
    "findings indicate that compact representations can maintain competitive detection power while "
    "substantially reducing computational latency."
)

output = predict_document(sample_academic_text)
print(json.dumps(output, indent=2))
